In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [2]:
df=pd.read_csv("winequality-red.csv",sep=";")
df.head(4)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [4]:
df["quality"].value_counts()
# imbalanced dataset - unequal classes

quality
5    681
6    638
7    199
4     53
8     18
3     10
Name: count, dtype: int64

In [5]:
features=df.drop("quality",axis=1)
target=df["quality"]

In [6]:
from sklearn.model_selection import train_test_split
xtrain,xtest,ytrain,ytest=train_test_split(features,target,random_state=0,
                                          test_size=0.2,stratify=target)
print(xtrain.shape,ytrain.shape)
print(xtest.shape,ytest.shape)

(1279, 11) (1279,)
(320, 11) (320,)


In [7]:
xtest[ytest==3]

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
1299,7.6,1.58,0.00,2.1,0.137,5.0,9.0,0.99476,3.50,0.40,10.9
459,11.6,0.58,0.66,2.2,0.074,10.0,47.0,1.00080,3.25,0.57,9.0


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report,confusion_matrix

In [9]:
def mymodel(model):
    model.fit(xtrain,ytrain)
    ypred=model.predict(xtest)
    # overfitting underfitting check
    print(f"Training Score : {model.score(xtrain,ytrain)}")
    print(f"Testing Score : {model.score(xtest,ytest)}")
    print(pd.DataFrame(confusion_matrix(ytest,ypred),
                   columns=[3,4,5,6,7,8],
                   index=[3,4,5,6,7,8])
     )
    print(classification_report(ytest,ypred))
    return model

In [10]:
dt=mymodel(DecisionTreeClassifier())

Training Score : 1.0
Testing Score : 0.625
   3  4   5   6   7  8
3  0  0   2   0   0  0
4  0  1   5   4   1  0
5  0  5  94  31   6  0
6  0  0  28  85  14  1
7  0  0   3  17  20  0
8  0  0   0   1   2  0
              precision    recall  f1-score   support

           3       0.00      0.00      0.00         2
           4       0.17      0.09      0.12        11
           5       0.71      0.69      0.70       136
           6       0.62      0.66      0.64       128
           7       0.47      0.50      0.48        40
           8       0.00      0.00      0.00         3

    accuracy                           0.62       320
   macro avg       0.33      0.32      0.32       320
weighted avg       0.61      0.62      0.62       320



In [11]:
parameters={
    "max_depth":list(range(2,8)),
    "min_samples_split":list(range(3,9)), 
    "min_samples_leaf":list(range(3,9)) 
    }


In [12]:
from sklearn.model_selection import GridSearchCV
clf=GridSearchCV(DecisionTreeClassifier(),parameters,verbose=2)
clf.fit(xtrain,ytrain)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=4; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=4; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=4; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=4; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=4; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=5; total time=   0.0s
[CV] END max_depth=2, min_samples_

GridSearchCV(estimator=DecisionTreeClassifier(),
             param_grid={'max_depth': [2, 3, 4, 5, 6, 7],
                         'min_samples_leaf': [3, 4, 5, 6, 7, 8],
                         'min_samples_split': [3, 4, 5, 6, 7, 8]},
             verbose=2)

In [13]:
clf.best_estimator_

DecisionTreeClassifier(max_depth=4, min_samples_leaf=6, min_samples_split=3)

In [14]:
c=mymodel(clf.best_estimator_)

Training Score : 0.6200156372165755
Testing Score : 0.578125
   3  4   5   6   7  8
3  0  0   0   2   0  0
4  0  0   3   7   1  0
5  0  0  95  38   3  0
6  0  0  43  73  12  0
7  0  0   4  19  17  0
8  0  0   0   1   2  0
              precision    recall  f1-score   support

           3       0.00      0.00      0.00         2
           4       0.00      0.00      0.00        11
           5       0.66      0.70      0.68       136
           6       0.52      0.57      0.54       128
           7       0.49      0.42      0.45        40
           8       0.00      0.00      0.00         3

    accuracy                           0.58       320
   macro avg       0.28      0.28      0.28       320
weighted avg       0.55      0.58      0.56       320



In [15]:
rf=mymodel(RandomForestClassifier())

Training Score : 1.0
Testing Score : 0.703125
   3  4    5   6   7  8
3  0  0    1   1   0  0
4  0  0    5   6   0  0
5  0  0  112  21   3  0
6  0  0   26  93   9  0
7  0  0    0  20  20  0
8  0  0    0   2   1  0
              precision    recall  f1-score   support

           3       0.00      0.00      0.00         2
           4       0.00      0.00      0.00        11
           5       0.78      0.82      0.80       136
           6       0.65      0.73      0.69       128
           7       0.61      0.50      0.55        40
           8       0.00      0.00      0.00         3

    accuracy                           0.70       320
   macro avg       0.34      0.34      0.34       320
weighted avg       0.67      0.70      0.68       320



In [16]:
# hyperparameter tuning of RandomForest
parameters={
    "n_estimators":[50,75,125], # number of trees in the forest, by default 100
    "max_depth":list(range(2,5)),
    "min_samples_split":list(range(3,5)), 
    "min_samples_leaf":list(range(3,5)) 
    }


In [17]:
clf_rf=GridSearchCV(RandomForestClassifier(),parameters,verbose=2)
clf_rf.fit(xtrain,ytrain)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3, n_estimators=50; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3, n_estimators=50; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3, n_estimators=50; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3, n_estimators=50; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3, n_estimators=50; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3, n_estimators=75; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3, n_estimators=75; total time=   0.1s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3, n_estimators=75; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3, n_estimators=75; total time=   0.1s
[CV] END max_depth=2, min_samples_leaf=3, min_s

GridSearchCV(estimator=RandomForestClassifier(),
             param_grid={'max_depth': [2, 3, 4], 'min_samples_leaf': [3, 4],
                         'min_samples_split': [3, 4],
                         'n_estimators': [50, 75, 125]},
             verbose=2)

In [19]:
clf_rf.best_estimator_

RandomForestClassifier(max_depth=4, min_samples_leaf=3, min_samples_split=4,
                       n_estimators=75)

In [20]:
gscv_rf=mymodel(clf_rf.best_estimator_)

Training Score : 0.6411258795934324
Testing Score : 0.596875
   3  4    5   6  7  8
3  0  0    1   1  0  0
4  0  0    3   8  0  0
5  0  0  100  36  0  0
6  0  0   40  85  3  0
7  0  0    2  32  6  0
8  0  0    0   3  0  0
              precision    recall  f1-score   support

           3       0.00      0.00      0.00         2
           4       0.00      0.00      0.00        11
           5       0.68      0.74      0.71       136
           6       0.52      0.66      0.58       128
           7       0.67      0.15      0.24        40
           8       0.00      0.00      0.00         3

    accuracy                           0.60       320
   macro avg       0.31      0.26      0.26       320
weighted avg       0.58      0.60      0.56       320



In [ ]:
# random over sampling - SMOTE(Synthetic Minority 
# Over-Sampling Technique)

In [21]:
pip install imbalanced-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [26]:
from imblearn.over_sampling import SMOTE

In [27]:
s=SMOTE(random_state=1)
xsample,ysample=s.fit_resample(xtrain,ytrain)

In [28]:
ysample.value_counts()

quality
5    545
6    545
7    545
4    545
8    545
3    545
Name: count, dtype: int64

In [30]:
def mymodel(model):
    model.fit(xsample,ysample)
    ypred=model.predict(xtest)
    # overfitting underfitting check
    print(f"Training Score : {model.score(xsample,ysample)}")
    print(f"Testing Score : {model.score(xtest,ytest)}")
    print(pd.DataFrame(confusion_matrix(ytest,ypred),
                   columns=[3,4,5,6,7,8],
                   index=[3,4,5,6,7,8])
     )
    print(classification_report(ytest,ypred))
    return model

In [31]:
gscv_rf_smote=mymodel(clf_rf.best_estimator_)

Training Score : 0.6993883792048929
Testing Score : 0.371875
   3   4   5   6   7   8
3  1   0   1   0   0   0
4  4   5   1   0   1   0
5  5  33  80  10   5   3
6  7  16  32  16  37  20
7  0   3   2   2  16  17
8  0   0   0   0   2   1
              precision    recall  f1-score   support

           3       0.06      0.50      0.11         2
           4       0.09      0.45      0.15        11
           5       0.69      0.59      0.63       136
           6       0.57      0.12      0.21       128
           7       0.26      0.40      0.32        40
           8       0.02      0.33      0.05         3

    accuracy                           0.37       320
   macro avg       0.28      0.40      0.24       320
weighted avg       0.56      0.37      0.40       320



In [32]:
df["quality"].value_counts()

quality
5    681
6    638
7    199
4     53
8     18
3     10
Name: count, dtype: int64

In [33]:
def multi_to_bin(c):
    if c<6:
        return 0
    else:
        return 1

In [34]:
df["quality_bin"]=df["quality"].apply(multi_to_bin)

In [36]:
df["quality_bin"].value_counts()

quality_bin
1    855
0    744
Name: count, dtype: int64

In [40]:
features=df.drop(["quality_bin","quality"],axis=1)
target=df["quality_bin"]

In [41]:
xtrain,xtest,ytrain,ytest=train_test_split(features,target,
                                          random_state=2,test_size=0.2)

In [42]:
def mymodel(model):
    model.fit(xtrain,ytrain)
    ypred=model.predict(xtest)
    # overfitting underfitting check
    print(f"Training Score : {model.score(xtrain,ytrain)}")
    print(f"Testing Score : {model.score(xtest,ytest)}")
    print(pd.DataFrame(confusion_matrix(ytest,ypred),
                   columns=["Bad","Good"],
                   index=["Bad","Good"])
     )
    print(classification_report(ytest,ypred))
    return model

In [44]:
clf_bin=GridSearchCV(RandomForestClassifier(),parameters,verbose=2)
clf_bin.fit(xtrain,ytrain)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3, n_estimators=50; total time=   0.1s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3, n_estimators=50; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3, n_estimators=50; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3, n_estimators=50; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3, n_estimators=50; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3, n_estimators=75; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3, n_estimators=75; total time=   0.1s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3, n_estimators=75; total time=   0.0s
[CV] END max_depth=2, min_samples_leaf=3, min_samples_split=3, n_estimators=75; total time=   0.1s
[CV] END max_depth=2, min_samples_leaf=3, min_s

GridSearchCV(estimator=RandomForestClassifier(),
             param_grid={'max_depth': [2, 3, 4], 'min_samples_leaf': [3, 4],
                         'min_samples_split': [3, 4],
                         'n_estimators': [50, 75, 125]},
             verbose=2)

In [45]:
final_model=mymodel(clf_bin.best_estimator_)

Training Score : 0.799061767005473
Testing Score : 0.728125
      Bad  Good
Bad    95    37
Good   50   138
              precision    recall  f1-score   support

           0       0.66      0.72      0.69       132
           1       0.79      0.73      0.76       188

    accuracy                           0.73       320
   macro avg       0.72      0.73      0.72       320
weighted avg       0.73      0.73      0.73       320

